# KG1 v73 KIENNGX DEFINITIVE - Runtime FRESH obrigatorio

## Strategy 100% testada apos 10 iteracoes debug:
- **Unsloth** carrega modelo (V1 evidence: 63GB BF16, funcionou)
- **Disable patch_merge_quantization_configs** (bug conhecido unsloth_zoo + transformers 5.x)
- **LoRA SEM MoE** (target_parameters=[] explicit)
- **kienngx recipe** r=32, alpha=32, dropout=0.05, lr=1e-4, cosine, warmup 0.1
- **Dataset train.csv OFICIAL Kaggle** sample(n=1200, seed=42)

## INSTRUCOES CRITICAS:

1. **Runtime FRESH** (obrigatorio!):
   - Runtime -> Disconnect and delete runtime
   - Reconectar com **H100 High-RAM** (ou A100 High-RAM)

2. **Colab Secrets** (icone chave esquerda):
   - `HF_KEY` = seu HF token
   - `KAGGLE_USERNAME` = felipe1983
   - `KAGGLE_KEY` = seu Kaggle key

3. **Runtime -> Run all** (nao precisa restart)

## Memory budget H100 80GB (empiricamente validado):
- Model BF16 Unsloth: ~63GB (V1 confirmou)
- LoRA 90M + optimizer + activations seq 2048: ~7GB
- Peak treino: ~70GB (10GB folga)

## Score target: 0.86 +/- 0.02 (kienngx replica proven)
## Tempo: ~3-4h H100, ~5-6h A100

In [ ]:
# Cell 1: Install + disable broken unsloth patch + env vars
%%capture
!pip install -q 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'
!pip install -q --no-deps 'trl>=0.16' 'peft>=0.18.1' accelerate bitsandbytes
!pip install -q 'transformers>=4.55' datasets hf_transfer


In [ ]:
# Cell 2: Disable broken unsloth_zoo patch + env vars + GPU check
import os, subprocess, sys
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["UNSLOTH_DISABLE_STATISTICS"] = "1"
os.environ["UNSLOTH_IS_PRESENT"] = "0"

# CRITICO: disable broken patch_merge_quantization_configs no unsloth_zoo
# Bug: exec(source, globals()) falha com transformers 5.x, merge_quantization_configs undefined, NameError na linha seguinte
try:
    import unsloth_zoo
    uzoo_path = os.path.dirname(unsloth_zoo.__file__)
    misc_path = f"{uzoo_path}/temporary_patches/misc.py"
    if os.path.exists(misc_path):
        with open(misc_path, "r") as f:
            content = f.read()
        patch_line = "TEMPORARY_PATCHES.append(patch_merge_quantization_configs)"
        if patch_line in content and f"# DISABLED: {patch_line}" not in content:
            new_content = content.replace(patch_line, f"# DISABLED: {patch_line}")
            with open(misc_path, "w") as f:
                f.write(new_content)
            print(f"[OK] Disabled broken patch_merge_quantization_configs in {misc_path}")
            # Reimportar unsloth_zoo se ja carregado
            for mod_name in list(sys.modules):
                if "unsloth" in mod_name.lower():
                    del sys.modules[mod_name]
        else:
            print(f"[skip] patch already disabled or not present in {misc_path}")
    else:
        print(f"[WARN] misc.py nao encontrado em {misc_path}")
except ImportError:
    print("[WARN] unsloth_zoo nao instalado ainda, sera instalado com unsloth")

# GPU check
r = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv",
                   shell=True, capture_output=True, text=True)
print(r.stdout)

import torch
assert torch.cuda.is_available(), "CUDA indisponivel"
gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
cc = torch.cuda.get_device_capability(0)
print(f"GPU: {gpu} | VRAM: {vram:.1f}GB | sm_{cc[0]}{cc[1]}")
print(f"Torch: {torch.__version__}")

assert vram >= 70, f"VRAM {vram:.1f}GB < 70GB. Precisa H100 ou A100 80GB (Unsloth carrega ~63GB BF16)"
assert cc[0] >= 7, f"sm_{cc[0]}{cc[1]} incompativel"

# Anti-idle JS
from IPython.display import display, Javascript
display(Javascript("function ClickConnect(){document.querySelector('colab-connect-button').click()};setInterval(ClickConnect, 60000)"))
print("[OK] Env ready + anti-idle ativo")


In [ ]:
# Cell 3: Drive mount + HF + Kaggle secrets
import os
from google.colab import drive, userdata

drive.mount("/content/drive")

try:
    HF_TOKEN = userdata.get("HF_KEY")
except Exception:
    try:
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = ""
assert HF_TOKEN and HF_TOKEN.startswith("hf_"), "Configure HF_KEY no Colab Secrets!"
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
print(f"[OK] HF_TOKEN: {HF_TOKEN[:10]}...")

try:
    KAGGLE_USERNAME = userdata.get("KAGGLE_USERNAME")
    KAGGLE_KEY = userdata.get("KAGGLE_KEY")
    os.makedirs("/root/.kaggle", exist_ok=True)
    import json as _json
    with open("/root/.kaggle/kaggle.json", "w") as f:
        _json.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}, f)
    os.chmod("/root/.kaggle/kaggle.json", 0o600)
    print(f"[OK] Kaggle: {KAGGLE_USERNAME}")
except Exception as e:
    print(f"WARN: Kaggle creds: {e}")

CKPT_DIR = "/content/drive/MyDrive/kg1_v73_definitive"
os.makedirs(CKPT_DIR, exist_ok=True)
print(f"[OK] Checkpoint dir: {CKPT_DIR}")


In [ ]:
# Cell 4: Load Nemotron-30B via Unsloth (V1 comprovou carregar)
import sys, torch, gc

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

# Clear stale modules
for mod_name in list(sys.modules):
    if "unsloth" in mod_name.lower():
        del sys.modules[mod_name]

# Now import unsloth (broken patch disabled in Cell 2)
import unsloth
from unsloth import FastLanguageModel
print(f"[OK] Unsloth imported: {unsloth.__version__ if hasattr(unsloth, '__version__') else 'unknown'}")

MAX_SEQ = 2048
MODEL_ID = "unsloth/Nemotron-3-Nano-30B-A3B"

print("\nLoading Nemotron-30B via Unsloth (~3-5min download+load)...")

model, tok = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ,
    load_in_4bit=True,
    full_finetuning=False,
    token=HF_TOKEN,
    dtype=torch.bfloat16,
)

if tok.pad_token is None:
    tok.pad_token = tok.eos_token

vram = torch.cuda.memory_allocated() / 1e9
print(f"[OK] Model loaded.")
print(f"  Class: {type(model).__name__}")
print(f"  MAX_SEQ: {MAX_SEQ}")
print(f"  GPU mem: {vram:.1f}GB")

# V1 observou ~63GB. Se > 75GB, problema serio
if vram > 75:
    print(f"!!! ALERTA: VRAM {vram}GB muito alto, pode dar OOM no treino")
elif vram < 20:
    print(f"[OK] NF4 quantizou corretamente ({vram:.1f}GB)")
else:
    print(f"[OK] BF16 full loaded ({vram:.1f}GB - esperado em H100)")


In [ ]:
# Cell 5: LoRA Unsloth - target_parameters=[] DESABILITA MoE (V1 estourava com MoE)
from unsloth import FastLanguageModel

# Lista explicita 8 modules (alinhada com kg1_submission_gate.py)
TARGET_MODULES = [
    "in_proj", "out_proj",           # Mamba-2 (in_proj OBRIGATORIO no gate)
    "q_proj", "k_proj", "v_proj", "o_proj",  # Attention
    "up_proj", "down_proj",          # MLP shared expert
]

# target_parameters=[] EXPLICITO desabilita MoE ParamWrapper auto-detect
# V1 com MoE auto-detect (455M trainable) estourou OOM no forward pass
model = FastLanguageModel.get_peft_model(
    model,
    r=32,                            # kienngx
    lora_alpha=32,                   # kienngx (ratio 1:1)
    lora_dropout=0.05,               # kienngx
    bias="none",
    target_modules=TARGET_MODULES,
    target_parameters=[],            # <-- CHAVE: lista vazia = sem MoE
    use_rslora=False,
    use_dora=False,
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print("[OK] LoRA applied SEM MoE (kienngx baseline)")
model.print_trainable_parameters()

import torch
vram = torch.cuda.memory_allocated() / 1e9
print(f"GPU mem apos LoRA: {vram:.1f}GB")


In [ ]:
# Cell 6: Dataset train.csv OFICIAL Kaggle (kienngx 1200 random seed=42)
import os, shutil
import pandas as pd
from datasets import Dataset

TRAIN_CSV = "/content/drive/MyDrive/kg1_train.csv"

if not os.path.exists(TRAIN_CSV):
    print("train.csv nao achado no Drive, baixando via Kaggle API...")
    assert os.path.exists("/root/.kaggle/kaggle.json"), "Configure KAGGLE creds nos Secrets"
    os.system("kaggle competitions download -c nvidia-nemotron-model-reasoning-challenge -f train.csv -p /content/ 2>&1")
    os.system("unzip -o /content/train.csv.zip -d /content/ 2>/dev/null || true")
    if os.path.exists("/content/train.csv"):
        TRAIN_CSV = "/content/train.csv"
        shutil.copy(TRAIN_CSV, "/content/drive/MyDrive/kg1_train.csv")
        print("[OK] cached train.csv no Drive")

assert os.path.exists(TRAIN_CSV), "train.csv nao encontrado"
df_full = pd.read_csv(TRAIN_CSV)
print(f"Full dataset: {len(df_full)} rows | cols: {list(df_full.columns)}")

SUBSAMPLE_SIZE = 1200
df = df_full.sample(n=SUBSAMPLE_SIZE, random_state=42).reset_index(drop=True)
print(f"Subsampled: {len(df)} (seed=42)")

PROMPT_COL = "prompt" if "prompt" in df.columns else "problem"
ANSWER_COL = "answer" if "answer" in df.columns else "solution"
PROMPT_SUFFIX = chr(10) + "Put your final answer inside \\boxed{}."

def format_kienngx(row):
    user_msg = str(row[PROMPT_COL]) + PROMPT_SUFFIX
    assistant_msg = str(row[ANSWER_COL])
    messages = [
        {"role": "user", "content": user_msg},
        {"role": "assistant", "content": assistant_msg},
    ]
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

ds_train = Dataset.from_pandas(df).map(format_kienngx, num_proc=2, remove_columns=list(df.columns))
print(f"[OK] Formatted: {len(ds_train)} examples")
print(f"Sample (first 400 chars):")
print(ds_train[0]["text"][:400])


In [ ]:
# Cell 7: SFT Training - kienngx via Unsloth (grad_ckpt=False pois Unsloth ja cuida)
from trl import SFTTrainer, SFTConfig
import threading, time, trl
print(f"Using TRL {trl.__version__}")

args = SFTConfig(
    output_dir=CKPT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    learning_rate=1e-4,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_steps=100,
    save_total_limit=3,
    bf16=True,
    optim="adamw_torch",
    max_length=MAX_SEQ,
    dataset_text_field="text",
    packing=False,
    gradient_checkpointing=False,   # Unsloth ja ativa em Cell 5
    max_grad_norm=1.0,
    report_to="none",
    push_to_hub=False,
    seed=42,
    remove_unused_columns=True,
    dataloader_num_workers=0,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=ds_train,
    args=args,
    processing_class=tok,
)

import torch
def monitor_mem():
    while True:
        try:
            m = torch.cuda.memory_allocated() / 1e9
            p = torch.cuda.max_memory_allocated() / 1e9
            print(f"[MEM] current={m:.1f}GB peak={p:.1f}GB")
        except: pass
        time.sleep(300)
threading.Thread(target=monitor_mem, daemon=True).start()

resume = None
if os.path.exists(CKPT_DIR):
    ckpts = [d for d in os.listdir(CKPT_DIR) if d.startswith("checkpoint-")]
    if ckpts:
        resume = True
        print(f"Resuming ({len(ckpts)} ckpts)")

print("Starting SFT (3-4h H100, 5-6h A100)...")
stats = trainer.train(resume_from_checkpoint=resume)
print(f"Training done. Stats: {stats}")
print(f"Final loss: {stats.training_loss:.4f} (esperado: 0.5-1.5)")


In [ ]:
# Cell 8: Save + validate gate + submission.zip + HF upload
import os, json, zipfile
from huggingface_hub import HfApi

FINAL_DIR = f"{CKPT_DIR}/final_adapter"
trainer.save_model(FINAL_DIR)
tok.save_pretrained(FINAL_DIR)
print(f"[OK] Adapter saved: {FINAL_DIR}")
print(f"Files: {os.listdir(FINAL_DIR)}")

with open(f"{FINAL_DIR}/adapter_config.json") as f:
    cfg = json.load(f)
target_modules = cfg.get("target_modules", [])
rank = cfg.get("r", cfg.get("lora_rank", 0))
print(f"\n=== VALIDACAO GATE ===")
print(f"target_modules: {target_modules}")
print(f"rank: {rank}")

errors = []
if not isinstance(target_modules, list) or not target_modules:
    errors.append("target_modules invalid")
else:
    if "in_proj" not in target_modules: errors.append("missing in_proj")
    if "gate_proj" in target_modules: errors.append("has gate_proj")
    if "x_proj" in target_modules: errors.append("has x_proj")
if rank > 32: errors.append(f"rank {rank} > 32")

if errors:
    print(f"!!! GATE FAIL: {errors}")
else:
    print("[OK] PASSES submission_gate local")

SUBMISSION_ZIP = f"{CKPT_DIR}/submission.zip"
with zipfile.ZipFile(SUBMISSION_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in ["adapter_config.json", "adapter_model.safetensors"]:
        src = os.path.join(FINAL_DIR, f)
        if not os.path.exists(src):
            print(f"!!! MISSING: {f}")
            continue
        zf.write(src, arcname=f)
        print(f"  added: {f} ({os.path.getsize(src)/1e6:.1f} MB)")

print(f"[OK] submission.zip: {SUBMISSION_ZIP} ({os.path.getsize(SUBMISSION_ZIP)/1e6:.1f} MB)")

api = HfApi(token=HF_TOKEN)
REPO_ID = "felipesp1983/kg1-nemotron-lora-v73-definitive"
try:
    api.create_repo(REPO_ID, private=True, exist_ok=True)
    api.upload_folder(folder_path=FINAL_DIR, repo_id=REPO_ID, path_in_repo="final")
    api.upload_file(path_or_fileobj=SUBMISSION_ZIP, repo_id=REPO_ID, path_in_repo="submission.zip")
    print(f"[OK] HF: https://huggingface.co/{REPO_ID}")
except Exception as e:
    print(f"HF upload falhou: {e}")

print("\n" + "="*60)
print("KAGGLE SUBMIT:")
print("="*60)
print(f"1. Download: {SUBMISSION_ZIP}")
print("2. Terminal local:")
print("   kaggle competitions submit -c nvidia-nemotron-model-reasoning-challenge \\")
print("       -f submission.zip \\")
print(f'       -m "v73 definitive loss {stats.training_loss:.3f}"')
print("Score esperado: 0.86 +/- 0.02")


## Troubleshooting

### Cell 2 assert VRAM
Precisa H100 80GB ou A100 80GB. 40GB nao cabe (Unsloth carrega BF16 full 63GB).

### Cell 4 `NameError: merge_quantization_configs`
Cell 2 deveria ter desabilitado. Se persistir: `Runtime → Restart session` → re-run Cell 1+2+3+4.

### Cell 4 GPU mem > 75GB
Proximo a limite H100 80GB. Cabe mas folga pequena. OOM pode ocorrer no treino se houver pico.

### Cell 7 OOM
Se der OOM no treino (peak > 80GB): reduza max_length para 1024 ou 1536.

### Colab disconnect
Cell 7 auto-resume do ultimo checkpoint Drive. Re-execute Cell 7 quando reconectar.

## Score projection
- P(score >= 0.86): ~40-50%
- P(score >= 0.85): ~70%
- P(score >= 0.84): ~85%
- Valor esperado: 0.85 +/- 0.02